In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')


def load_data():
    try:
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
        cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
                'thalach','exang','oldpeak','slope','ca','thal','target']
        df = pd.read_csv(url, header=None, names=cols, na_values='?')
        df.dropna(inplace=True)
        df['target'] = (df['target'] > 0).astype(int)
        return df
    except:
        return generate_data()


def generate_data(n=1000):
    np.random.seed(42)
    half = n // 2

    def samples(n, risk):
        age    = np.random.normal(60 if risk else 45, 10, n).clip(29, 77)
        sex    = np.random.binomial(1, 0.7 if risk else 0.4, n)
        cp     = np.random.choice([0,1,2,3], n, p=[0.5,0.2,0.2,0.1] if risk else [0.1,0.3,0.3,0.3])
        tbps   = np.random.normal(145 if risk else 120, 18, n).clip(90, 200)
        chol   = np.random.normal(260 if risk else 210, 45, n).clip(130, 400)
        fbs    = np.random.binomial(1, 0.3 if risk else 0.1, n)
        recg   = np.random.choice([0,1,2], n)
        thal   = np.random.normal(140 if risk else 165, 22, n).clip(70, 200)
        exang  = np.random.binomial(1, 0.6 if risk else 0.15, n)
        oldpk  = np.random.exponential(1.8 if risk else 0.5, n).clip(0, 6.2)
        slope  = np.random.choice([0,1,2], n)
        ca     = np.random.choice([0,1,2,3], n, p=[0.2,0.3,0.3,0.2] if risk else [0.6,0.2,0.1,0.1])
        thalv  = np.random.choice([1,2,3], n)
        target = np.ones(n, int) if risk else np.zeros(n, int)
        return np.column_stack([age,sex,cp,tbps,chol,fbs,recg,thal,exang,oldpk,slope,ca,thalv,target])

    data = np.vstack([samples(half, True), samples(n - half, False)])
    np.random.shuffle(data)
    cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
            'thalach','exang','oldpeak','slope','ca','thal','target']
    return pd.DataFrame(data, columns=cols)


# Bat Algorithm for feature selection
class BatAlgorithm:
    def _init_(self, n_bats=20, n_iter=50, random_state=42):
        self.n_bats = n_bats
        self.n_iter = n_iter
        self.random_state = random_state
        self.best_features_ = None
        self.best_score_ = 0.0

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def _fitness(self, X, y, mask):
        if mask.sum() == 0:
            return 0.0
        clf = LogisticRegression(max_iter=300, random_state=42)
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        scores = cross_val_score(clf, X[:, mask], y, cv=cv, scoring='accuracy')
        return scores.mean() - 0.001 * mask.sum() / len(mask)

    def select_features(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n_feat = X.shape[1]

        positions  = rng.integers(0, 2, (self.n_bats, n_feat)).astype(float)
        velocities = rng.uniform(-1, 1, (self.n_bats, n_feat))
        loudness   = rng.uniform(0.5, 1.0, self.n_bats)
        pulse_rate = rng.uniform(0.0, 0.5, self.n_bats)

        scores = np.array([self._fitness(X, y, positions[i] > 0.5) for i in range(self.n_bats)])
        best_idx = np.argmax(scores)
        best_pos = positions[best_idx].copy()
        best_score = scores[best_idx]

        for t in range(self.n_iter):
            for i in range(self.n_bats):
                freq = 0.0 + 2.0 * rng.random()
                velocities[i] += (positions[i] - best_pos) * freq
                new_pos = positions[i] + velocities[i]

                if rng.random() > pulse_rate[i]:
                    noise = loudness.mean() * rng.uniform(-1, 1, n_feat) * 0.1
                    new_pos = best_pos + noise

                binary = self._sigmoid(new_pos) > 0.5
                new_score = self._fitness(X, y, binary)

                if new_score > scores[i] and rng.random() < loudness[i]:
                    positions[i] = new_pos
                    scores[i] = new_score
                    loudness[i] *= 0.9
                    pulse_rate[i] *= (1 - np.exp(-0.9 * t))

                if new_score > best_score:
                    best_pos = new_pos.copy()
                    best_score = new_score

        self.best_features_ = self._sigmoid(best_pos) > 0.5
        self.best_score_ = best_score
        return self.best_features_


# Fuzzy rules for interpretability (ANFIS-inspired)
class FuzzyRuleEngine:
    rules = [
        ("age",      lambda v: v >= 60,  "Age >= 60"),
        ("trestbps", lambda v: v >= 140, "BP >= 140"),
        ("chol",     lambda v: v >= 240, "Cholesterol >= 240"),
        ("thalach",  lambda v: v <= 120, "Max Heart Rate <= 120"),
        ("oldpeak",  lambda v: v >= 2.0, "ST Depression >= 2.0"),
        ("fbs",      lambda v: v == 1,   "Fasting Blood Sugar high"),
        ("exang",    lambda v: v == 1,   "Exercise Angina present"),
        ("ca",       lambda v: v >= 2,   "Vessels blocked >= 2"),
    ]

    def predict_risk(self, row):
        fired = []
        for col, condition, label in self.rules:
            if col in row and condition(row[col]):
                fired.append(label)
        score = len(fired)
        if score <= 2:
            risk = "Low"
        elif score <= 4:
            risk = "Medium"
        else:
            risk = "High"
        reasoning = "IF " + " AND ".join(fired) + f" THEN risk is {risk}" if fired else "No critical markers found"
        return {"risk_level": risk, "score": score, "fired_rules": fired, "reasoning": reasoning}


def build_model():
    base = [
        ('lr',  LogisticRegression(C=1.0, max_iter=500, random_state=42)),
        ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=500,
                              early_stopping=True, random_state=42)),
        ('rf',  RandomForestClassifier(n_estimators=200, random_state=42)),
        ('gbm', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                           max_depth=4, random_state=42)),
        ('svm', SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)),
    ]
    return StackingClassifier(estimators=base,
                               final_estimator=LogisticRegression(max_iter=300),
                               cv=5, stack_method='predict_proba', n_jobs=-1)


def predict_patient(patient_dict, model, scaler, selected_mask, feature_names, fuzzy):
    row = [patient_dict.get(f, 0) for f in feature_names]
    X = scaler.transform(np.array(row).reshape(1, -1))[:, selected_mask]
    prob = model.predict_proba(X)[0, 1]
    fuzzy_res = fuzzy.predict_risk(patient_dict)
    combined = 0.7 * prob + 0.3 * (fuzzy_res["score"] / 8)
    if combined < 0.35:
        risk = "Low"
    elif combined < 0.65:
        risk = "Medium"
    else:
        risk = "High"
    return {
        "probability": round(float(prob), 4),
        "combined_score": round(float(combined), 4),
        "risk_level": risk,
        "reasoning": fuzzy_res["reasoning"],
        "fired_rules": fuzzy_res["fired_rules"]
    }


if _name_ == "_main_":
    df = load_data()
    print(f"Dataset: {df.shape}, Target distribution: {df['target'].value_counts().to_dict()}")

    feature_names = df.drop('target', axis=1).columns.tolist()
    X = df.drop('target', axis=1).values
    y = df['target'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    print("Running Bat Algorithm for feature selection...")
    ba = BatAlgorithm(n_bats=20, n_iter=40, random_state=42)
    mask = ba.select_features(X_scaled, y)
    selected = [feature_names[i] for i, s in enumerate(mask) if s]
    print(f"Selected features ({len(selected)}): {selected}")

    X_sel = X_scaled[:, mask]
    X_train, X_test, y_train, y_test = train_test_split(
        X_sel, y, test_size=0.2, random_state=42, stratify=y)

    print("\nTraining stacking ensemble...")
    model = build_model()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\nAccuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))